In [1]:
# Cell 1: Google Drive mount (a CPU runtime is enough for Stage 5)
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# Cell 1b: Label-fraction files on Drive: results, predictions and checkpoint folders
from pathlib import Path
import json

CONTENT = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content")

print("Results files (frac10):")
for f in sorted((CONTENT / "results").glob("*frac10*")):
    print("   ", f.name)

print("\nPrediction files (frac10):")
for f in sorted((CONTENT / "predictions").glob("*frac10*")):
    print("   ", f.name)

print("\nCheckpoint folders (frac10): files and last saved epoch")
for d in sorted((CONTENT / "checkpoints").glob("*frac10*")):
    files = sorted(p.name for p in d.iterdir())
    history = d / "history.json"
    last_epoch = json.loads(history.read_text())[-1]["epoch"] if history.exists() else None
    print(f"    {d.name}: {files}, last epoch = {last_epoch}")

Results files (frac10):
    prithvi_frozen_frac10_seed0.json
    subset_frac10.json
    unet_s2_6_frac10_seed0.json
    unet_s2_6_frac10_seed1.json

Prediction files (frac10):
    prithvi_frozen_frac10_seed0_bolivia.npz
    unet_s2_6_frac10_seed0_bolivia.npz
    unet_s2_6_frac10_seed1_bolivia.npz

Checkpoint folders (frac10): files and last saved epoch
    prithvi_frozen_frac10_seed0: ['best.pt', 'history.json', 'last.pt'], last epoch = 100
    prithvi_frozen_frac10_seed1: ['best.pt', 'history.json', 'last.pt'], last epoch = 72
    prithvi_frozen_frac10_seed2: [], last epoch = None
    unet_s2_6_frac10_seed0: ['best.pt', 'history.json', 'last.pt'], last epoch = 100
    unet_s2_6_frac10_seed1: ['best.pt', 'history.json', 'last.pt'], last epoch = 100
    unet_s2_6_frac10_seed2: [], last epoch = None


 Cell 2 reads all 24 results files and puts the TP, FP and FN of each chip into arrays, with the same chip order for all methods.

In [4]:
# Cell 2: Results files, method list and per-chip counts in a fixed chip order
import json
from pathlib import Path
import numpy as np
import pandas as pd

RESULTS_DIR = Path("/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content/results")
SPLITS = ["test", "bolivia"]

# Method key: (label, results files of its seeds)
METHODS = {
    "otsu": ("Otsu (VH)", ["otsu_vh"]),
    "fixed": ("Fixed VH threshold", ["fixed_vh"]),
    "unet_s1": ("U-Net radar, 2 bands", [f"unet_s1_seed{s}" for s in range(3)]),
    "unet_s2": ("U-Net optical, 13 bands", [f"unet_s2_seed{s}" for s in range(3)]),
    "unet_s2_6": ("U-Net optical, 6 bands", [f"unet_s2_6_seed{s}" for s in range(3)]),
    "prithvi_frozen": ("Prithvi frozen, 6 bands", [f"prithvi_frozen_seed{s}" for s in range(3)]),
    "prithvi_full": ("Prithvi full, 6 bands", ["prithvi_full_seed0"]),
    "unet_s2_6_frac10": ("U-Net optical, 6 bands, 10% labels", [f"unet_s2_6_frac10_seed{s}" for s in range(3)]),
    "prithvi_frozen_frac10": ("Prithvi frozen, 6 bands, 10% labels", [f"prithvi_frozen_frac10_seed{s}" for s in range(3)]),
}

results = {name: json.loads((RESULTS_DIR / f"{name}.json").read_text())
           for _, files in METHODS.values() for name in files}
print("Results files read:", len(results))

# Fixed chip order for each split; every results file must contain exactly these chips
CHIPS = {split: sorted(results["fixed_vh"]["splits"][split]["chips"]) for split in SPLITS}
for name, r in results.items():
    for split in SPLITS:
        assert sorted(r["splits"][split]["chips"]) == CHIPS[split], f"Chip mismatch: {name}, {split}"

def counts_array(name, split):
    """Per-chip TP, FP, FN of one results file, shape (n_chips, 3), in the fixed chip order."""
    chips = results[name]["splits"][split]["chips"]
    return np.array([[chips[c]["tp"], chips[c]["fp"], chips[c]["fn"]] for c in CHIPS[split]], dtype=np.int64)

# Counts of each method and split, shape (n_seeds, n_chips, 3)
COUNTS = {key: {split: np.stack([counts_array(n, split) for n in files]) for split in SPLITS}
          for key, (_, files) in METHODS.items()}

for split in SPLITS:
    print(f"{split}: {len(CHIPS[split])} chips")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/2026Research/FloodFM-Sen1Floods11/content/results/unet_s2_6_frac10_seed2.json'